In [24]:
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'using devices : {device}')

using devices : cpu


# 데이터셋

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1


In [26]:
import os

csv_path = os.path.join(path, 'IMDB Dataset.csv')
df = pd.read_csv(csv_path)

print(df.columns.tolist())
print(df['sentiment'].value_counts())
display(df.head())

['review', 'sentiment']
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [27]:
df = df.rename(columns={'review': 'text', 'sentiment': 'label'})

# CPU이므로 샘플만 사용 (원하는 크기로 조절)
df_sample = df.sample(n=500, random_state=42).reset_index(drop=True)

train_df = df_sample.sample(frac=0.8, random_state=42).reset_index(drop=True)
test_df = df_sample.drop(train_df.index).reset_index(drop=True)

# 기존 evaluate_model 함수가 dict list를 받으므로 변환
train_data = train_df.to_dict('records')
test_data = test_df.to_dict('records')

print(f'Train: {len(train_data)}, Test: {len(test_data)}')

Train: 400, Test: 100


# 모델 로드 및 모델 매개변수(Parameter) 분석
Hugging Face의 transformers 라이브러리를 활용하여 구글이 공개한 지시어 미세조정(Instruction-tuned) 모델인 google/flan-t5-small을 로드합니다. 또한 파인튜닝 전에 모델의 총 매개변수(Total Parameters) 와 학습 가능한 매개변수(Trainable Parameters) 를 확인합니다.

In [28]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [29]:
# _, param = next(iter(model.named_parameters()))

# 파라미터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라미터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)

In [30]:
# 추론 수행 함수
def generate_prediction(prompt, model, tokenizer,max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
    return pred_text

# 평가 루프
def evaluate_model(model, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0
    results=[]

    # few shot 예시
    few_shot_examples = (
        "Review: The cinematography was beautiful, and the story was touching.\nSentiment:positive\n\n"
        "Review: Horrible acting and bad direction. Do not watch this film.\nSentiment:negative\n\n"
        "Review: Brilliant performance by the lead actor, a must-watch.\nSentiment:positive\n\n"
    )

    for item in test_data:
        text = item['text']
        true_label = item['label']

        if prompt_type =='zero_shot':
            prompt=f'Review:{text}\nSentiment (positive or negative)'
        elif prompt_type =='few_shot':
            prompt=few_shot_examples + f'Review:{text}\nSentiment (positive or negative)'
        else:
            prompt=f'Review:{text}\nSentiment (positive or negative)'
        pred_label = generate_prediction(prompt,model,tokenizer)
        # 긍정/부정 단어 클랜징
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label
        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1
        results.append({
            'Text':text, 'True Label' : true_label, 
            'Pred Label':pred_label, 'Cleaned Pred' : cleaned_pred, 'Correct':is_correct
        })
        
    accuracy = correct / len(test_data)
    return accuracy, pd.DataFrame(results)

# Zero Shot Programming

In [31]:
# 비교용으로 일부만 평가
sample_test = test_data[:20]
before_acc, before_df = evaluate_model(model, tokenizer, sample_test, 'zero_shot')
print(f'[파인튜닝 전] Zero Shot Accuracy: {before_acc:.2%}')
display(before_df)

[파인튜닝 전] Zero Shot Accuracy: 45.00%


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The actors were not believable, The story was ...",negative,the movie was a waste of time and money,the movie was a waste of time and money,False
1,please save your money and go see something el...,negative,this movie is a waste of time and money,this movie is a waste of time and money,False
2,I never really watched this program before alt...,positive,the bbc is a great show to watch.,the bbc is a great show to watch.,False
3,I own a copy of this film and have always love...,positive,positive,positive,True
4,I hadn't seen this in many years. The acting w...,negative,positive,positive,False
5,This movie was an amazing tribute to whoever h...,positive,this movie is a great tribute to the people,this movie is a great tribute to the people,False
6,Return to Cabin by the Lake just.... was lacki...,negative,negative,negative,True
7,A remake of a successful movie can be a tricky...,negative,"'s portrayed' is, it'","'s portrayed' is, it'",False
8,Andie McDowell is beautiful as the 40-ish woma...,negative,negative,negative,True
9,I agree with another user here and have to say...,positive,this is a great movie for kids. it,this is a great movie for kids. it,False


# 전체 매개변수 파인튜닝 (Full Fine Tuning)

In [32]:
# 1. 데이터 토크나이징 함수 - 전처리

def preprocess_function(example):
    # T5 모델에 맞게 토큰화
    inputs = [f"Review: {text}\nSentiment: Answer with either positive or negative." for text in example['text']]
    model_input = tokenizer(inputs, max_length=128, truncation=True)

    # 라벨을 토큰화
    labels = tokenizer(text_target=example['label'], max_length=128, truncation=True)
    model_input['labels'] = labels['input_ids']
    return model_input
    
# 2. DataSet 객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑
tokenized_train = train_dataset.map(preprocess_function, batched= True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched = True, remove_columns=train_dataset.column_names)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [33]:
# training Arguments 설정

training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_sentiment_results',
    eval_strategy='epoch',
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=2
)

# Data Collator, Trainer 객체
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model= model,
    args=training_args,
    train_dataset = tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

# 학습 시간 측정
start_time = time.time()
trainer.train()
training_time = time.time() - start_time
print(f'Full Fine-tuning completed : {training_time:.2f} seconds')


c:\Users\Playdata\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.474112,0.258472
2,0.091312,0.137860
3,0.065642,0.101459
4,0.252264,0.106368
5,0.006650,0.099298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Playdata\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Full Fine-tuning completed : 393.17 seconds


# 파인튜닝된 모델 최종 평가

In [34]:
after_acc, after_df = evaluate_model(model, tokenizer, sample_test, 'fine_tuned')
print(f'[파인튜닝 전] Accuracy: {before_acc:.2%}')
print(f'[파인튜닝 후] Accuracy: {after_acc:.2%}')
display(after_df)


[파인튜닝 전] Accuracy: 45.00%
[파인튜닝 후] Accuracy: 85.00%


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The actors were not believable, The story was ...",negative,negative,negative,True
1,please save your money and go see something el...,negative,negative,negative,True
2,I never really watched this program before alt...,positive,positive,positive,True
3,I own a copy of this film and have always love...,positive,positive,positive,True
4,I hadn't seen this in many years. The acting w...,negative,positive,positive,False
5,This movie was an amazing tribute to whoever h...,positive,positive,positive,True
6,Return to Cabin by the Lake just.... was lacki...,negative,negative,negative,True
7,A remake of a successful movie can be a tricky...,negative,negative,negative,True
8,Andie McDowell is beautiful as the 40-ish woma...,negative,negative,negative,True
9,I agree with another user here and have to say...,positive,positive,positive,True
